# Experiment 2 (v3) — Learned predictors: MLP, LSTM, StaticGNN, ST-GNN
**Paper:** Learning-based residual link lifetime prediction for UAV swarms (*Drones*, MDPI)

**v3 fixes (code audit):** causal link state; full 60-s follow-up for every label; KF-LET tuned on validation
(12 settings); 6 test episodes; single-slot (batch = 1) latency measured separately from batched throughput;
checkpoints and training logs saved.

**Before running:** `Runtime → Change runtime type → T4 GPU`.

* `SAVE_TO_DRIVE = True` (recommended): results are written to Google Drive after **every** model.
  If Colab disconnects, just open the notebook again and `Run all` — finished models are skipped.
* Runtime ≈ 2–2.5 h on a T4. At the end **`exp2_results.zip`** is downloaded — send it to me.

In [ ]:
%%writefile uav_swarm_sim.py
"""
UAV Swarm Link-Lifetime Dataset Generator
=========================================
Paper: Learning-Based Residual Link Lifetime Prediction for UAV Swarms
Author: Renat Abdirayimov (TATU)
Version: 2 (causal link state, fully observed lifetime labels)

What this module does
---------------------
1. Moves N UAVs in 3D using three mobility models:
   - Gauss-Markov (GM): each UAV flies independently, smooth random motion
   - RPGM (Reference Point Group Mobility): UAVs fly in groups (swarm squads)
   - RWP (3D random waypoint): straight flights between random waypoints
2. Computes the air-to-air (A2A) channel for every UAV pair:
   path loss + time-correlated shadowing + Rician small-scale fading -> SNR
3. Decides which links are UP (large-scale SNR >= threshold) -> a dynamic graph.
   The link state is CAUSAL: it depends only on the current and past slots.
4. Labels every UP link with its TRUE residual lifetime (seconds until it breaks).
   The episode is simulated for T + H seconds, but samples are taken only in
   [0, T), so every label has a full H-second follow-up.
5. Computes the classic analytical baseline: Link Expiration Time (LET)
"""
import numpy as np

# ----------------------------------------------------------------------------
# 1. Simulation parameters (these go into Table I of the paper)
# ----------------------------------------------------------------------------
DEFAULT_CFG = dict(
    n_uav=30,              # number of UAVs
    area=(1500.0, 1500.0), # x, y size [m]
    alt=(50.0, 150.0),     # min/max altitude [m]
    dt=0.5,                # time step [s]
    T=600.0,               # episode length [s]
    mobility="rpgm",       # "gm", "rpgm" or "rwp"
    group_size=5,          # UAVs per group (RPGM)
    v_mean=12.0,           # mean speed [m/s]
    v_max=35.0,            # max UAV speed [m/s]
    gm_alpha=0.85,         # Gauss-Markov memory (0=random walk, 1=straight line)
    group_radius=250.0,    # max deviation of a member from its group centre [m]
    member_alpha=0.95,     # memory of a member's own motion inside the group
    member_noise=2.0,      # random acceleration of members [m/s per step]
    member_pull=0.002,     # spring force pulling members back to the centre
    # ---- channel (A2A link) ----
    fc=2.4e9,              # carrier frequency [Hz]
    pt_dbm=10.0,           # transmit power [dBm]
    bw=20e6,               # bandwidth [Hz]
    nf_db=7.0,             # receiver noise figure [dB]
    pl_exp=2.2,            # A2A path-loss exponent
    shadow_std=2.0,        # shadowing std [dB]
    shadow_dcorr=50.0,     # shadowing decorrelation distance [m]
    shadow_model="link",   # "link": independent per link (Gudmundson in distance change)
                           # "spatial": static 2-D shadowing field shared by all links
    field_corr=100.0,      # correlation length of the spatial shadowing field [m]
    rician_k_db=7.6,       # Rician K-factor, measured A2A value (Ad Hoc Networks 2025)
    snr_th_db=10.0,        # link is UP if large-scale SNR >= this [dB]
    break_steps=2,         # link is broken only if down for >= 2 steps (1 s)
    horizon=60.0,          # residual lifetime is capped (censored) here [s]
    channel="a2a",         # "a2a" (SNR threshold, shadowing + Rician) or "disk"
    rwp_pause_max=2.0,     # RWP pause time at a waypoint, U(0, max) [s]
    obs_pos_std=2.0,       # GPS position error in Hello messages [m]
    obs_vel_std=0.3,       # velocity error in Hello messages [m/s]
    seed=0,
)

C = 3e8


def noise_dbm(cfg):
    return -174.0 + 10 * np.log10(cfg["bw"]) + cfg["nf_db"]


def path_loss_db(d, cfg):
    """Log-distance path loss with free-space reference at 1 m."""
    pl0 = 20 * np.log10(4 * np.pi * cfg["fc"] / C)  # FSPL at 1 m
    return pl0 + 10 * cfg["pl_exp"] * np.log10(np.maximum(d, 1.0))


def comm_range(cfg):
    """Distance where mean SNR (no shadowing) equals the threshold."""
    pl_max = cfg["pt_dbm"] - noise_dbm(cfg) - cfg["snr_th_db"]
    pl0 = 20 * np.log10(4 * np.pi * cfg["fc"] / C)
    return 10 ** ((pl_max - pl0) / (10 * cfg["pl_exp"]))


# ----------------------------------------------------------------------------
# 2. Mobility models
# ----------------------------------------------------------------------------
def _reflect(pos, vel, mean_dir, lo, hi):
    """Bounce UAVs off the borders of the area (velocity and mean direction flip)."""
    for k in range(3):
        below, above = pos[:, k] < lo[k], pos[:, k] > hi[k]
        pos[below, k] = 2 * lo[k] - pos[below, k]
        pos[above, k] = 2 * hi[k] - pos[above, k]
        vel[below | above, k] *= -1
        mean_dir[below | above, k] *= -1
    return pos, vel, mean_dir


def _gauss_markov_step(vel, mean_dir, cfg, rng):
    a = cfg["gm_alpha"]
    noise = rng.normal(0, 1, vel.shape) * cfg["v_mean"] * 0.5
    noise[:, 2] *= 0.2  # vertical motion is smaller
    return a * vel + (1 - a) * mean_dir + np.sqrt(1 - a ** 2) * noise


def simulate_mobility(cfg, rng):
    """Returns positions and velocities, shape (n_steps, n_uav, 3)."""
    n, dt = cfg["n_uav"], cfg["dt"]
    steps = int(cfg["T"] / dt)
    lo = np.array([0, 0, cfg["alt"][0]])
    hi = np.array([cfg["area"][0], cfg["area"][1], cfg["alt"][1]])

    def rand_heading(m):
        th = rng.uniform(0, 2 * np.pi, m)
        v = np.stack([np.cos(th), np.sin(th), np.zeros(m)], 1) * cfg["v_mean"]
        return v

    P = np.zeros((steps, n, 3))
    V = np.zeros((steps, n, 3))

    if cfg["mobility"] == "gm":
        pos = rng.uniform(lo, hi, (n, 3))
        vel = rand_heading(n)
        mean_dir = vel.copy()
        for t in range(steps):
            P[t], V[t] = pos, vel
            vel = _gauss_markov_step(vel, mean_dir, cfg, rng)
            pos = pos + vel * dt
            pos, vel, mean_dir = _reflect(pos, vel, mean_dir, lo, hi)
    elif cfg["mobility"] == "rpgm":
        g = cfg["group_size"]
        n_groups = int(np.ceil(n / g))
        member_group = np.arange(n) // g
        margin = np.array([200.0, 200.0, 0.0])
        c_pos = rng.uniform(lo + margin, hi - margin, (n_groups, 3))
        c_pos[:, 2] = rng.uniform(*cfg["alt"], n_groups)
        c_vel = rand_heading(n_groups)
        c_mean = c_vel.copy()
        # each member keeps a random offset that slowly drifts (Gauss-Markov)
        offset = rng.normal(0, cfg["group_radius"] / 2, (n, 3))
        offset[:, 2] *= 0.3
        off_vel = np.zeros((n, 3))
        prev = c_pos[member_group] + offset
        for t in range(steps):
            pos = np.clip(c_pos[member_group] + offset, lo, hi)
            if t > 0:  # physical speed limit (removes jumps at the borders)
                step = pos - prev
                sp = np.linalg.norm(step, axis=1, keepdims=True) / dt
                pos = prev + np.where(sp > cfg["v_max"], step * cfg["v_max"] / np.maximum(sp, 1e-9), step)
                vel = (pos - prev) / dt
            else:
                vel = c_vel[member_group].copy()
            P[t], V[t] = pos, vel
            prev = pos
            # group centres move with Gauss-Markov
            c_vel = _gauss_markov_step(c_vel, c_mean, cfg, rng)
            c_pos = c_pos + c_vel * dt
            c_pos, c_vel, c_mean = _reflect(c_pos, c_vel, c_mean, lo, hi)
            # members wander inside the group (pulled back to the centre)
            off_vel = cfg["member_alpha"] * off_vel + rng.normal(0, cfg["member_noise"], (n, 3)) \
                - cfg["member_pull"] * offset
            off_vel[:, 2] *= 0.5
            offset = offset + off_vel * dt
            r = np.linalg.norm(offset, axis=1, keepdims=True)
            offset = np.where(r > cfg["group_radius"], offset * cfg["group_radius"] / r, offset)
    elif cfg["mobility"] == "rwp":
        # 3-D random waypoint: fly straight to a random waypoint at a random
        # speed, pause, pick the next waypoint
        pos = rng.uniform(lo, hi, (n, 3))
        wp = rng.uniform(lo, hi, (n, 3))
        speed = rng.uniform(0.5, 1.5, n) * cfg["v_mean"]
        pause = np.zeros(n)
        for t in range(steps):
            d = wp - pos
            dist = np.linalg.norm(d, axis=1)
            moving = pause <= 0
            vel = np.where(moving[:, None], d / np.maximum(dist, 1e-9)[:, None] * speed[:, None], 0.0)
            arrive = moving & (dist <= speed * dt)
            vel[arrive] = d[arrive] / dt
            P[t], V[t] = pos, vel
            pos = pos + vel * dt
            pause = pause - dt
            # arrivals: pause, then new waypoint and speed
            pause[arrive] = rng.uniform(0, cfg["rwp_pause_max"], arrive.sum())
            wp[arrive] = rng.uniform(lo, hi, (arrive.sum(), 3))
            speed[arrive] = rng.uniform(0.5, 1.5, arrive.sum()) * cfg["v_mean"]
    else:
        raise ValueError("mobility must be 'gm', 'rpgm' or 'rwp'")
    return P, V


# ----------------------------------------------------------------------------
# 3. A2A channel
# ----------------------------------------------------------------------------
def simulate_channel(P, cfg, rng):
    """Returns distance, large-scale SNR and measured (faded) SNR for all pairs."""
    steps, n, _ = P.shape
    iu, ju = np.triu_indices(n, 1)
    diff = P[:, iu, :] - P[:, ju, :]
    dist = np.linalg.norm(diff, axis=2)                       # (steps, n_pairs)

    if cfg.get("shadow_model", "link") == "spatial":
        shadow = spatial_shadowing(P, iu, ju, cfg, rng)
        return _finish_channel(dist, shadow, iu, ju, cfg, rng)
    # time-correlated shadowing (Gudmundson model): rho = exp(-moved / d_corr)
    shadow = np.zeros_like(dist)
    shadow[0] = rng.normal(0, cfg["shadow_std"], dist.shape[1])
    for t in range(1, steps):
        moved = np.abs(dist[t] - dist[t - 1]) + 1e-3
        rho = np.exp(-moved / cfg["shadow_dcorr"])
        shadow[t] = rho * shadow[t - 1] + np.sqrt(1 - rho ** 2) * rng.normal(
            0, cfg["shadow_std"], dist.shape[1])

    return _finish_channel(dist, shadow, iu, ju, cfg, rng)


def spatial_shadowing(P, iu, ju, cfg, rng, res=10.0, n_pts=8):
    """Spatially correlated shadowing: a static 2-D Gaussian random field G(x, y)
    with correlation length `field_corr` (buildings / terrain). The shadowing of
    link (i, j) is the field averaged along the ground projection of the segment
    i-j, rescaled to std `shadow_std`. Links that cross the same region are
    therefore correlated, and a link's shadowing changes as its endpoints move."""
    from scipy.ndimage import gaussian_filter
    nx = int(cfg["area"][0] / res) + 1
    ny = int(cfg["area"][1] / res) + 1
    G = gaussian_filter(rng.normal(size=(ny, nx)), sigma=cfg["field_corr"] / res / 2, mode="wrap")
    G = (G - G.mean()) / G.std()
    s = np.linspace(0.0, 1.0, n_pts)
    pi, pj = P[:, iu, :2], P[:, ju, :2]                          # (T, M, 2)
    pts = pi[:, :, None, :] + s[None, None, :, None] * (pj - pi)[:, :, None, :]
    ix = np.clip(np.rint(pts[..., 0] / res).astype(int), 0, nx - 1)
    iy = np.clip(np.rint(pts[..., 1] / res).astype(int), 0, ny - 1)
    L = G[iy, ix].mean(-1)                                       # (T, M)
    return cfg["shadow_std"] * L / L.std()


def _finish_channel(dist, shadow, iu, ju, cfg, rng):
    snr_ls = cfg["pt_dbm"] - path_loss_db(dist, cfg) - shadow - noise_dbm(cfg)

    # Rician fading power |h|^2 with unit mean
    K = 10 ** (cfg["rician_k_db"] / 10)
    los = np.sqrt(K / (K + 1))
    nlos = np.sqrt(1 / (2 * (K + 1)))
    h = los + nlos * (rng.normal(size=dist.shape) + 1j * rng.normal(size=dist.shape))
    snr_meas = snr_ls + 10 * np.log10(np.abs(h) ** 2)
    return dist, snr_ls, snr_meas, iu, ju


# ----------------------------------------------------------------------------
# 4. Labels: true residual link lifetime + LET baseline
# ----------------------------------------------------------------------------
def debounce(up_raw, k):
    """CAUSAL link state machine (uses only the current and past slots).
    An active link is declared BROKEN at the k-th consecutive slot whose
    large-scale SNR is below the threshold (k missed Hello messages); during
    the first k-1 low slots it is still active. A broken link becomes active
    again as soon as the threshold is exceeded. Initial state = up_raw[0]."""
    if k <= 1:
        return up_raw.copy()
    out = np.zeros_like(up_raw, dtype=bool)
    state = up_raw[0].copy()
    miss = np.where(up_raw[0], 0, k).astype(int)
    out[0] = state
    for t in range(1, up_raw.shape[0]):
        miss = np.where(up_raw[t], 0, miss + 1)
        state = np.where(up_raw[t], True, state & (miss < k))
        out[t] = state
    return out


def residual_lifetime(up, cfg):
    """For every (t, pair) that is UP, seconds until the link first goes DOWN.
    Capped at `horizon` (right-censored). Returns (lifetime, censored_flag)."""
    steps, m = up.shape
    dt, H = cfg["dt"], cfg["horizon"]
    life = np.zeros((steps, m))
    nxt = np.full(m, np.inf)  # steps until next DOWN, scanning backwards
    for t in range(steps - 1, -1, -1):
        nxt = np.where(up[t], nxt + 1, 0)
        life[t] = nxt
    life = life * dt
    censored = life >= H
    life = np.minimum(life, H)
    life[~up] = 0.0
    return life, censored & up


def link_expiration_time(P, V, iu, ju, r, cfg):
    """Classic LET (Su, Lee & Gerla, 2001) extended to 3D: assumes both UAVs
    keep their current velocity. Solves |dp + dv*tau| = r for tau > 0."""
    dp = P[:, iu, :] - P[:, ju, :]
    dv = V[:, iu, :] - V[:, ju, :]
    a = np.sum(dv * dv, -1)
    b = 2 * np.sum(dp * dv, -1)
    c = np.sum(dp * dp, -1) - r ** 2
    disc = np.maximum(b ** 2 - 4 * a * c, 0)
    with np.errstate(divide="ignore", invalid="ignore"):
        tau = (-b + np.sqrt(disc)) / (2 * a)
    tau = np.where(a < 1e-9, cfg["horizon"], tau)
    tau = np.where(c > 0, 0.0, tau)  # already out of range
    return np.clip(tau, 0, cfg["horizon"])


# ----------------------------------------------------------------------------
# 5. One full episode -> dictionary of arrays
# ----------------------------------------------------------------------------
def generate_episode(cfg=None, **overrides):
    cfg = {**DEFAULT_CFG, **(cfg or {}), **overrides}
    rng = np.random.default_rng(cfg["seed"])
    n_keep = int(round(cfg["T"] / cfg["dt"]))            # sample slots [0, T)
    sim_cfg = {**cfg, "T": cfg["T"] + cfg["horizon"] + cfg["dt"]}   # + follow-up
    P, V = simulate_mobility(sim_cfg, rng)
    dist, snr_ls, snr_meas, iu, ju = simulate_channel(P, cfg, rng)
    if cfg["channel"] == "disk":          # classical distance-threshold link
        up_raw = dist <= comm_range(cfg)
    elif cfg["channel"] == "a2a":         # SNR threshold with shadowing
        up_raw = snr_ls >= cfg["snr_th_db"]
    else:
        raise ValueError("channel must be 'a2a' or 'disk'")
    up = debounce(up_raw, cfg["break_steps"])
    life, cens = residual_lifetime(up, cfg)
    # Hello-message observations: GPS position/velocity errors
    P_obs = P + rng.normal(0, cfg["obs_pos_std"], P.shape)
    V_obs = V + rng.normal(0, cfg["obs_vel_std"], V.shape)
    let = link_expiration_time(P_obs, V_obs, iu, ju, comm_range(cfg), cfg)
    # every kept label has a full horizon of follow-up inside the simulation
    assert up.shape[0] >= n_keep + int(round(cfg["horizon"] / cfg["dt"])) + 1
    cut = lambda x: x[:n_keep]
    P, V, P_obs, V_obs, dist, snr_ls, snr_meas, up, life, cens, let = map(
        cut, (P, V, P_obs, V_obs, dist, snr_ls, snr_meas, up, life, cens, let))
    assert np.isfinite(life).all()
    return dict(cfg=cfg, pos=P, vel=V, pos_obs=P_obs, vel_obs=V_obs, dist=dist,
                snr_ls=snr_ls, snr_meas=snr_meas, up=up, life=life, censored=cens,
                let=let, iu=iu, ju=ju)


def self_test(verbose=True):
    """Unit tests for the two properties checked in the code audit."""
    k = 2
    f = lambda seq: debounce(np.array(seq, bool)[:, None], k)[:, 0].astype(int).tolist()
    assert f([1, 0, 1]) == [1, 1, 1]          # 1-slot dip: still active
    assert f([1, 0, 0]) == [1, 1, 0]          # break declared at the 2nd miss
    assert f([1, 0, 0, 1]) == [1, 1, 0, 1]    # reconnection
    assert f([0, 0, 1]) == [0, 0, 1]          # starts down
    # causality: same history, different futures -> identical states so far
    rng = np.random.default_rng(1)
    for _ in range(200):
        a = rng.random((40, 5)) > 0.3
        b = a.copy(); t = rng.integers(1, 39); b[t + 1:] = rng.random((39 - t, 5)) > 0.3
        assert (debounce(a, k)[:t + 1] == debounce(b, k)[:t + 1]).all()
    # labels: every sample has a full follow-up; censored <=> life == H
    ep = generate_episode(mobility="gm", seed=123, T=60.0)
    H = ep["cfg"]["horizon"]
    assert ep["up"].shape[0] == 120
    assert ((ep["life"] >= H) == ep["censored"])[ep["up"]].all()
    # a label at t must not depend on anything after t + H (finite follow-up)
    cfg = {**DEFAULT_CFG, "dt": 0.5, "horizon": 5.0}
    nH = 10
    for _ in range(50):
        u = rng.random((80, 4)) > 0.1
        full, _ = residual_lifetime(u, cfg)
        for t in rng.integers(0, 80 - nH - 1, 5):
            part, _ = residual_lifetime(u[:t + nH + 1], cfg)
            assert np.allclose(full[t], part[t])
    if verbose:
        print("self_test passed: causal link state, full-horizon labels")


if __name__ == "__main__":
    ep = generate_episode()
    cfg = ep["cfg"]
    up = ep["up"]
    print(f"Comm range R = {comm_range(cfg):.1f} m, noise = {noise_dbm(cfg):.1f} dBm")
    print(f"Steps = {up.shape[0]}, pairs = {up.shape[1]}")
    print(f"Avg active links per snapshot = {up.sum(1).mean():.1f}")
    print(f"Avg node degree = {2*up.sum(1).mean()/cfg['n_uav']:.2f}")
    m = up & ~ep["censored"]
    print(f"Uncensored samples = {m.sum()}, mean lifetime = {ep['life'][m].mean():.1f} s")
    err = np.abs(ep["let"][m] - ep["life"][m])
    print(f"LET baseline MAE = {err.mean():.2f} s")

In [ ]:
%%writefile baselines.py
"""
Analytical baselines and evaluation metrics for residual link lifetime.

- LET    : constant-velocity link expiration time (Su, Lee & Gerla, 2001)
- KF-LET : Kalman filter with a Singer (manoeuvring-target) acceleration model,
           as in Kalman-assisted FANET routing (MP-QGRD, KFQR); the filtered
           trajectories are extrapolated and the first time the distance
           exceeds the range R is the predicted lifetime.
"""
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score


# ---------------------------------------------------------------------------
# Kalman filter (Singer model), identical for every UAV and axis -> vectorised
# ---------------------------------------------------------------------------
def singer_matrices(dt, tau, q):
    a = 1.0 / tau
    e = np.exp(-a * dt)
    F = np.array([[1, dt, (a * dt - 1 + e) / a**2],
                  [0, 1, (1 - e) / a],
                  [0, 0, e]])
    Q = q * np.array([[dt**5 / 20, dt**4 / 8, dt**3 / 6],
                      [dt**4 / 8, dt**3 / 3, dt**2 / 2],
                      [dt**3 / 6, dt**2 / 2, dt]])
    return F, Q


def kalman_singer(P_obs, V_obs, cfg, tau=3.0, q=2.0):
    """Filter noisy positions/velocities. Inputs (T, N, 3).
    Returns filtered position, velocity, acceleration (T, N, 3)."""
    T, N, _ = P_obs.shape
    dt = cfg["dt"]
    F, Q = singer_matrices(dt, tau, q)
    H = np.array([[1.0, 0, 0], [0, 1.0, 0]])
    R = np.diag([cfg["obs_pos_std"]**2, cfg["obs_vel_std"]**2])
    x = np.zeros((N, 3, 3))                     # (uav, axis, state)
    x[:, :, 0], x[:, :, 1] = P_obs[0], V_obs[0]
    Pc = np.diag([cfg["obs_pos_std"]**2, cfg["obs_vel_std"]**2, 4.0])
    out = np.zeros((T, N, 3, 3))
    for t in range(T):
        if t > 0:                               # predict
            x = x @ F.T
            Pc = F @ Pc @ F.T + Q
        S = H @ Pc @ H.T + R                    # update (same gain for all)
        K = Pc @ H.T @ np.linalg.inv(S)
        z = np.stack([P_obs[t], V_obs[t]], -1)  # (N, 3, 2)
        x = x + (z - x @ H.T) @ K.T
        Pc = (np.eye(3) - K @ H) @ Pc
        out[t] = x
    return out[..., 0], out[..., 1], out[..., 2]


def kf_let(P_obs, V_obs, iu, ju, R, cfg, tau=3.0, q=2.0):
    """Predicted lifetime from Singer-model extrapolation of both UAVs."""
    p, v, a = kalman_singer(P_obs, V_obs, cfg, tau, q)
    H, dt = cfg["horizon"], cfg["dt"]
    tk = np.arange(dt, H + dt / 2, dt)                      # prediction grid
    al = 1.0 / tau
    g = (np.exp(-al * tk) - 1 + al * tk) / al**2            # Singer position gain
    dp, dv, da = p[:, iu] - p[:, ju], v[:, iu] - v[:, ju], a[:, iu] - a[:, ju]
    T, M, _ = dp.shape
    life = np.full((T, M), H)
    for t in range(T):
        traj = dp[t][None] + tk[:, None, None] * dv[t][None] + g[:, None, None] * da[t][None]
        out = np.linalg.norm(traj, axis=2) > R                # (K, M)
        first = np.where(out.any(0), out.argmax(0), -1)
        life[t] = np.where(first >= 0, tk[np.maximum(first, 0)], H)
        life[t][np.linalg.norm(dp[t], axis=1) > R] = 0.0
    return np.clip(life, 0, H)


# ---------------------------------------------------------------------------
# Metrics
# ---------------------------------------------------------------------------
def evaluate(pred, life, censored, up, delta=5.0):
    """Regression metrics on uncensored active links + break-within-delta
    classification on all active links (censored links never break early)."""
    m = up & ~censored
    y, yh = life[m], pred[m]
    err = yh - y
    mae, rmse = np.abs(err).mean(), np.sqrt((err**2).mean())
    r2 = 1 - (err**2).sum() / ((y - y.mean())**2).sum()
    lab = (up & ~censored & (life <= delta))[up]
    score = -pred[up]
    return dict(MAE=mae, RMSE=rmse, R2=r2,
                AUC=roc_auc_score(lab, score), AUPRC=average_precision_score(lab, score),
                n=int(m.sum()), pos_rate=lab.mean())

In [ ]:
%%writefile models.py
"""
Learned predictors of residual link lifetime (PyTorch).

All models receive, for a batch of snapshots t and an observation window of W slots:
  node : (B, W, N, Fn)   node features  (position, velocity, degree)
  edge : (B, W, M, Fe)   pair features  (distance, radial/relative speed, SNR, dSNR,
                                          capacity, LET, link-up flag), zero if link down
  up   : (B, W, M)       link state
and output the predicted residual lifetime of every pair at slot t: (B, M) in [0, H].

Models
  MLP        per-link, current slot only
  LSTM       per-link, sequence over the window (JPE / LPMD-GPSR style, no graph)
  StaticGNN  edge-aware GAT on the current snapshot only (no temporal model)
  STGNN      edge-aware GAT on every snapshot + node GRU + link GRU

Note: MLP and LSTM do not aggregate neighbour features, but their node
features include the local node degree.
"""
import torch
import torch.nn as nn
import torch.nn.functional as F


def mlp(i, h, o, depth=2, drop=0.1):
    layers, d = [], i
    for _ in range(depth):
        layers += [nn.Linear(d, h), nn.ReLU(), nn.Dropout(drop)]
        d = h
    layers.append(nn.Linear(d, o))
    return nn.Sequential(*layers)


def pair_node_feats(node, iu, ju):
    """Symmetric combination of the two endpoint feature vectors."""
    a, b = node[..., iu, :], node[..., ju, :]
    return torch.cat([a + b, (a - b).abs()], -1)


class EdgeGAT(nn.Module):
    """Dense edge-feature-aware graph attention layer (multi-head)."""

    def __init__(self, d, fe, heads=4):
        super().__init__()
        assert d % heads == 0
        self.h, self.dk = heads, d // heads
        self.W = nn.Linear(d, d, bias=False)
        self.U = nn.Linear(fe, d, bias=False)
        self.a_dst = nn.Parameter(0.1 * torch.randn(heads, self.dk))
        self.a_src = nn.Parameter(0.1 * torch.randn(heads, self.dk))
        self.a_e = nn.Parameter(0.1 * torch.randn(heads, self.dk))
        self.self_lin = nn.Linear(d, d)
        self.norm = nn.LayerNorm(d)

    def forward(self, h, E, A):
        # h (B, N, d) | E (B, N, N, fe) | A (B, N, N) bool
        B, N, d = h.shape
        Wh = self.W(h).view(B, N, self.h, self.dk)
        Ue = self.U(E).view(B, N, N, self.h, self.dk)
        sc = ((Wh * self.a_dst).sum(-1)[:, :, None, :]          # receiver i
              + (Wh * self.a_src).sum(-1)[:, None, :, :]        # sender j
              + (Ue * self.a_e).sum(-1))                        # link (i, j)
        sc = F.leaky_relu(sc, 0.2)
        mask = A.unsqueeze(-1)
        sc = sc.masked_fill(~mask, -1e4)
        att = torch.softmax(sc, dim=2) * mask                   # isolated node -> zero message
        msg = torch.einsum("bijh,bjhd->bihd", att, Wh) + (att.unsqueeze(-1) * Ue).sum(2)
        out = F.elu(self.self_lin(h) + msg.reshape(B, N, d))
        return self.norm(out + h)


class MLPModel(nn.Module):
    def __init__(self, fn, fe, H, d=128):
        super().__init__()
        self.H, self.net = H, mlp(fe + 2 * fn, d, 1, depth=3)

    def forward(self, node, edge, up, iu, ju):
        x = torch.cat([edge[:, -1], pair_node_feats(node[:, -1], iu, ju)], -1)
        return self.H * torch.sigmoid(self.net(x).squeeze(-1))


class LSTMModel(nn.Module):
    def __init__(self, fn, fe, H, d=64):
        super().__init__()
        self.H = H
        self.lstm = nn.LSTM(fe + 2 * fn, d, num_layers=2, batch_first=True, dropout=0.1)
        self.head = mlp(d, d, 1)

    def forward(self, node, edge, up, iu, ju):
        B, W, M, _ = edge.shape
        x = torch.cat([edge, pair_node_feats(node, iu, ju)], -1)       # (B, W, M, F)
        x = x.permute(0, 2, 1, 3).reshape(B * M, W, -1)
        _, (h, _) = self.lstm(x)
        return self.H * torch.sigmoid(self.head(h[-1]).view(B, M))


class STGNN(nn.Module):
    """Proposed model. temporal=False gives the StaticGNN ablation."""

    def __init__(self, fn, fe, H, n_nodes, iu, ju, d=64, layers=2, heads=4, dg=32, temporal=True):
        super().__init__()
        self.H, self.N, self.temporal = H, n_nodes, temporal
        self.register_buffer("iu", iu)
        self.register_buffer("ju", ju)
        self.enc = nn.Linear(fn, d)
        self.gat = nn.ModuleList([EdgeGAT(d, fe, heads) for _ in range(layers)])
        if temporal:
            self.node_gru = nn.GRU(d, d, batch_first=True)
            self.edge_gru = nn.GRU(fe, dg, batch_first=True)
        else:
            self.edge_mlp = mlp(fe, dg, dg, depth=1)
        self.head = mlp(2 * d + dg + fe, 128, 1, depth=2)

    def dense(self, edge, up):
        B, W, M, fe = edge.shape
        E = edge.new_zeros(B, W, self.N, self.N, fe)
        A = torch.zeros(B, W, self.N, self.N, dtype=torch.bool, device=edge.device)
        E[:, :, self.iu, self.ju] = edge
        E[:, :, self.ju, self.iu] = edge
        A[:, :, self.iu, self.ju] = up
        A[:, :, self.ju, self.iu] = up
        return E, A

    def forward(self, node, edge, up, iu=None, ju=None):
        if not self.temporal:                                   # current snapshot only
            node, edge, up = node[:, -1:], edge[:, -1:], up[:, -1:]
        B, W, N, _ = node.shape
        E, A = self.dense(edge, up)
        h = self.enc(node).reshape(B * W, N, -1)
        E, A = E.reshape(B * W, N, N, -1), A.reshape(B * W, N, N)
        for layer in self.gat:
            h = layer(h, E, A)
        h = h.view(B, W, N, -1)
        if self.temporal:
            _, z = self.node_gru(h.permute(0, 2, 1, 3).reshape(B * N, W, -1))
            z = z[-1].view(B, N, -1)
            M = edge.shape[2]
            _, g = self.edge_gru(edge.permute(0, 2, 1, 3).reshape(B * M, W, -1))
            g = g[-1].view(B, M, -1)
        else:
            z = h[:, -1]
            g = self.edge_mlp(edge[:, -1])
        zi, zj = z[:, self.iu], z[:, self.ju]
        x = torch.cat([zi + zj, (zi - zj).abs(), g, edge[:, -1]], -1)
        return self.H * torch.sigmoid(self.head(x).squeeze(-1))


def censored_huber(pred, y, cens, mask, H, delta=2.0, lam=1.0):
    """Huber loss on uncensored links; one-sided penalty (pred < H) on censored links.
    NOTE: because every model outputs H*sigmoid(.) <= H and censored targets are H,
    this loss is mathematically identical to plain_huber. Kept for reference only."""
    unc = F.huber_loss(pred, y, reduction="none", delta=delta)
    cen = F.huber_loss(torch.clamp(H - pred, min=0), torch.zeros_like(pred), reduction="none", delta=delta)
    loss = torch.where(cens, lam * cen, unc)
    m = mask.float()
    return (loss * m).sum() / m.sum().clamp(min=1.0)


def plain_huber(pred, y, cens, mask, H, delta=2.0, lam=1.0):
    """Huber loss over all active links; censored samples have target H (= lower bound)."""
    loss = F.huber_loss(pred, y, reduction="none", delta=delta)
    m = mask.float()
    return (loss * m).sum() / m.sum().clamp(min=1.0)

In [ ]:
%%writefile data_torch.py
"""Feature construction and batching for Experiment 2 (all tensors kept on the GPU)."""
import numpy as np
import torch
from uav_swarm_sim import generate_episode, comm_range

FN, FE = 7, 8   # node / edge feature sizes


def episode_features(ep):
    cfg = ep["cfg"]
    R, H = comm_range(cfg), cfg["horizon"]
    p, v, iu, ju = ep["pos_obs"], ep["vel_obs"], ep["iu"], ep["ju"]
    up = ep["up"]
    T, N, _ = p.shape
    dp, dv = p[:, iu] - p[:, ju], v[:, iu] - v[:, ju]
    dist = np.linalg.norm(dp, axis=2)
    radial = (dp * dv).sum(2) / np.maximum(dist, 1.0)
    rel = np.linalg.norm(dv, axis=2)
    snr = ep["snr_meas"]
    dsnr = np.diff(snr, axis=0, prepend=snr[:1])
    cap = np.log2(1 + 10 ** (np.clip(snr, -20, 60) / 10))
    ef = np.stack([dist / R, radial / 35, rel / 35, snr / 30, dsnr / 10, cap / 10,
                   ep["let"] / H, up.astype(float)], -1)
    ef[~up] = 0.0
    inc = np.zeros((len(iu), N))
    inc[np.arange(len(iu)), iu] = 1.0
    inc[np.arange(len(iu)), ju] = 1.0
    deg = up.astype(float) @ inc                     # node degree (T, N)
    nf = np.concatenate([p[..., :2] / cfg["area"][0], (p[..., 2:] - cfg["alt"][0]) / 100,
                         v / 35, deg[..., None] / N], -1)
    return nf.astype(np.float32), ef.astype(np.float32)


class EpisodeBank:
    """Generates episodes (same seeds as Experiment 1) and stores features on device."""

    def __init__(self, mobility, episodes, channel="a2a", device="cuda", seed0=10_000, **cfg_over):
        nodes, edges, ups, lifes, cens, lets = [], [], [], [], [], []
        self.raw = {}
        for s in episodes:
            ep = generate_episode(mobility=mobility, channel=channel, seed=seed0 + s, **cfg_over)
            nf, ef = episode_features(ep)
            nodes.append(nf); edges.append(ef); ups.append(ep["up"])
            lifes.append(ep["life"]); cens.append(ep["censored"]); lets.append(ep["let"])
            self.raw[s] = dict(pos_obs=ep["pos_obs"], vel_obs=ep["vel_obs"], cfg=ep["cfg"])
        self.cfg = ep["cfg"]
        self.H = float(self.cfg["horizon"])
        self.episodes = list(episodes)
        self.iu = torch.as_tensor(ep["iu"], device=device)
        self.ju = torch.as_tensor(ep["ju"], device=device)
        t = lambda x, dt: torch.as_tensor(np.stack(x), dtype=dt, device=device)
        self.node = t(nodes, torch.float32)          # (E, T, N, FN)
        self.edge = t(edges, torch.float16)          # (E, T, M, FE)  half precision to save memory
        self.up = t(ups, torch.bool)                 # (E, T, M)
        self.life = t(lifes, torch.float32)
        self.cens = t(cens, torch.bool)
        self.let = np.stack(lets)
        self.T, self.N, self.M = self.node.shape[1], self.node.shape[2], self.edge.shape[2]

    def index(self, W, stride=1):
        e = np.repeat(np.arange(len(self.episodes)), len(range(W - 1, self.T, stride)))
        t = np.tile(np.arange(W - 1, self.T, stride), len(self.episodes))
        return np.stack([e, t], 1)

    def batch(self, idx, W):
        dev = self.node.device
        e = torch.as_tensor(idx[:, 0], device=dev)
        t = torch.as_tensor(idx[:, 1], device=dev)
        ts = t[:, None] - torch.arange(W - 1, -1, -1, device=dev)[None]
        ee = e[:, None].expand_as(ts)
        return (self.node[ee, ts], self.edge[ee, ts].float(), self.up[ee, ts],
                self.life[e, t], self.cens[e, t], self.up[e, t])

In [ ]:
import numpy as np, pandas as pd, torch, time, json, os, itertools, copy
import matplotlib.pyplot as plt
import importlib, uav_swarm_sim, baselines, models, data_torch
for m in (uav_swarm_sim, baselines, models, data_torch): importlib.reload(m)
uav_swarm_sim.self_test()   # must print "self_test passed"
from uav_swarm_sim import comm_range
from baselines import evaluate, kf_let
from models import MLPModel, LSTMModel, STGNN, plain_huber
from data_torch import EpisodeBank, FN, FE

QUICK = False           # full experiment (set True only for a 5-min test)
SAVE_TO_DRIVE = True    # keep results in Google Drive -> resume after a disconnect
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEV, torch.cuda.get_device_name(0) if DEV == "cuda" else "")

W = 8                   # observation window: 8 slots = 4 s
MOBILITY = ["rpgm", "gm", "rwp"]
MODELS = ["MLP", "LSTM", "StaticGNN", "STGNN"]
SEEDS = [0, 1, 2]
EPOCHS, PATIENCE, BATCH, LR = 30, 6, 32, 1e-3
TRAIN_EPS, VAL_EPS, TEST_EPS = range(0, 14), range(14, 17), range(17, 23)
KF_GRID = list(itertools.product([2, 3, 5, 10], [0.5, 1.0, 2.0]))   # same 12 settings as Experiment 1
if QUICK:
    MOBILITY, SEEDS, EPOCHS = ["rpgm"], [0], 2
    TRAIN_EPS, VAL_EPS, TEST_EPS = range(0, 3), range(14, 15), range(17, 18)
    KF_GRID = [(2, 1.0)]

OUT = "exp2_out"
if SAVE_TO_DRIVE and not QUICK:
    from google.colab import drive; drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/uav_gnn_paper/exp2_v3_out"
for d in ("", "/runs", "/ckpt", "/logs"): os.makedirs(OUT + d, exist_ok=True)
print("results folder:", OUT)

In [ ]:
def build(name, bank):
    H = bank.H
    if name == "MLP":       return MLPModel(FN, FE, H)
    if name == "LSTM":      return LSTMModel(FN, FE, H)
    if name == "StaticGNN": return STGNN(FN, FE, H, bank.N, bank.iu, bank.ju, temporal=False)
    if name == "STGNN":     return STGNN(FN, FE, H, bank.N, bank.iu, bank.ju, temporal=True)
    raise ValueError(name)

@torch.no_grad()
def predict(model, bank, bs=64):
    model.eval()
    idx = bank.index(W, stride=1)
    pred = np.zeros((len(bank.episodes), bank.T, bank.M), np.float32)
    for k in range(0, len(idx), bs):
        b = idx[k:k + bs]
        node, edge, up, *_ = bank.batch(b, W)
        out = model(node, edge, up, bank.iu, bank.ju).float().cpu().numpy()
        pred[b[:, 0], b[:, 1]] = out
    return pred

@torch.no_grad()
def val_mae(model, bank):
    pred = predict(model, bank)
    life, cens, up = bank.life.cpu().numpy(), bank.cens.cpu().numpy(), bank.up.cpu().numpy()
    m = up[:, W-1:] & ~cens[:, W-1:]
    return np.abs(pred[:, W-1:][m] - life[:, W-1:][m]).mean()

def train(name, seed, tr, va, loss_fn=plain_huber, log=None):
    torch.manual_seed(seed); np.random.seed(seed)
    model = build(name, tr).to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=1)
    idx = tr.index(W, stride=1)
    best, best_state, bad, hist = np.inf, None, 0, []
    for epoch in range(EPOCHS):
        model.train(); t0 = time.time(); tot = 0.0
        perm = np.random.permutation(len(idx))
        for k in range(0, len(perm), BATCH):
            node, edge, up, y, c, m = tr.batch(idx[perm[k:k + BATCH]], W)
            loss = loss_fn(model(node, edge, up, tr.iu, tr.ju), y, c, m, tr.H)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            tot += loss.item()
        v = val_mae(model, va); sched.step(v)
        hist.append(dict(epoch=epoch + 1, train_loss=tot / (len(perm) / BATCH), val_MAE=v, sec=time.time() - t0))
        print(f"  {name:9s} seed {seed} epoch {epoch+1:2d}  loss {hist[-1]['train_loss']:.3f}  val MAE {v:.3f}  ({hist[-1]['sec']:.0f}s)")
        if v < best - 1e-3:
            best, best_state, bad = v, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= PATIENCE: break
    model.load_state_dict(best_state)
    if log: pd.DataFrame(hist).to_csv(f"{OUT}/logs/{log}.csv", index=False)
    return model, best

@torch.no_grad()
def latency_ms(model, bank, n=200, warm=20):
    # Online latency: ONE slot of the whole swarm (batch = 1), GPU synchronised.
    # Includes feature gathering on the GPU and the forward pass; excludes the simulator.
    model.eval(); idx = bank.index(W, stride=1)[:n + warm]; ts = []
    for k, b in enumerate(idx):
        if DEV == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter()
        node, edge, up, *_ = bank.batch(b[None], W)
        model(node, edge, up, bank.iu, bank.ju)
        if DEV == "cuda": torch.cuda.synchronize()
        if k >= warm: ts.append(1e3 * (time.perf_counter() - t0))
    return float(np.median(ts)), float(np.percentile(ts, 95))

def tune_kf(va):
    # Select (tau, q) of KF-LET on the validation episodes only.
    rows = []
    life, cens, up = va.life.cpu().numpy(), va.cens.cpu().numpy(), va.up.cpu().numpy()
    for tau, q in KF_GRID:
        maes = []
        for k, e in enumerate(va.episodes):
            p = kf_let(va.raw[e]["pos_obs"], va.raw[e]["vel_obs"], va.iu.cpu().numpy(), va.ju.cpu().numpy(),
                       comm_range(va.cfg), va.cfg, tau, q)
            maes.append(evaluate(p[W-1:], life[k, W-1:], cens[k, W-1:], up[k, W-1:])["MAE"])
        rows.append(dict(tau=tau, q=q, val_MAE=float(np.mean(maes))))
    t = pd.DataFrame(rows); b = t.loc[t.val_MAE.idxmin()]
    return float(b.tau), float(b.q), t

def score(pred, bank, method, mob, seed):
    rows = []
    life, cens, up = bank.life.cpu().numpy(), bank.cens.cpu().numpy(), bank.up.cpu().numpy()
    for k, e in enumerate(bank.episodes):
        r = evaluate(pred[k, W-1:], life[k, W-1:], cens[k, W-1:], up[k, W-1:])
        r.update(method=method, mobility=mob, seed=seed, episode=e); rows.append(r)
    return rows

## Main experiment: all models × mobility models × seeds

In [ ]:
rows, nparams, lat = [], {}, []
t_start = time.time()
for mob in MOBILITY:
    print(f"=== {mob.upper()} ===  building data ...")
    tr = EpisodeBank(mob, TRAIN_EPS, device=DEV)
    va = EpisodeBank(mob, VAL_EPS, device=DEV)
    te = EpisodeBank(mob, TEST_EPS, device=DEV)
    f_base = f"{OUT}/runs/{mob}_baselines.csv"
    if not os.path.exists(f_base):
        tau, q, kt = tune_kf(va); kt["mobility"] = mob; kt.to_csv(f"{OUT}/runs/{mob}_kf_tuning.csv", index=False)
        print(f"  KF-LET tuned on validation: tau={tau}, q={q}")
        kf = np.stack([kf_let(te.raw[e]["pos_obs"], te.raw[e]["vel_obs"], te.iu.cpu().numpy(), te.ju.cpu().numpy(),
                              comm_range(te.cfg), te.cfg, tau, q) for e in te.episodes])
        b = score(te.let.astype(np.float32), te, "LET", mob, 0) + score(kf.astype(np.float32), te, "KF-LET", mob, 0)
        pd.DataFrame(b).to_csv(f_base, index=False)
    rows += pd.read_csv(f_base).to_dict("records")
    for name, seed in itertools.product(MODELS, SEEDS):
        f_run = f"{OUT}/runs/{mob}_{name}_s{seed}.csv"
        if os.path.exists(f_run):                       # finished before a disconnect
            rows += pd.read_csv(f_run).to_dict("records"); print(f"  {name} seed {seed}: already done"); continue
        t0 = time.time()
        model, v = train(name, seed, tr, va, log=f"{mob}_{name}_s{seed}")
        nparams[name] = sum(p.numel() for p in model.parameters())
        torch.save(model.state_dict(), f"{OUT}/ckpt/{mob}_{name}_s{seed}.pt")
        t1 = time.time(); pred = predict(model, te)
        if DEV == "cuda": torch.cuda.synchronize()
        t_thr = (time.time() - t1) / (len(te.episodes) * (te.T - W + 1))
        med, p95 = latency_ms(model, te) if seed == SEEDS[0] else (np.nan, np.nan)
        r = score(pred, te, name, mob, seed)
        for x in r: x.update(train_s=t1 - t0, batched_ms_per_slot=1e3 * t_thr, latency_ms_median=med,
                             latency_ms_p95=p95, params=nparams[name], val_MAE=v)
        pd.DataFrame(r).to_csv(f_run, index=False); rows += r
        print(f"  -> {name} seed {seed}: test MAE {np.mean([x['MAE'] for x in r]):.3f}   (elapsed {(time.time()-t_start)/60:.1f} min)")
    del tr, va, te; torch.cuda.empty_cache()
res = pd.DataFrame(rows); res.to_csv(f"{OUT}/results_exp2.csv", index=False)
summary = (res.groupby(["mobility", "method"])[["MAE", "RMSE", "R2", "AUC", "AUPRC"]].agg(["mean", "std"]).round(3))
display(summary)

## Ablation (ST-GNN, RPGM): without SNR-based edge features

In [ ]:
if not QUICK:
    mob = "rpgm"
    tr = EpisodeBank(mob, TRAIN_EPS, device=DEV); va = EpisodeBank(mob, VAL_EPS, device=DEV); te = EpisodeBank(mob, TEST_EPS, device=DEV)
    keep = torch.ones(FE, device=DEV); keep[[3, 4, 5]] = 0        # SNR, dSNR, capacity
    for bank in (tr, va, te): bank.edge = (bank.edge.float() * keep).half()
    for seed in SEEDS:
        f_run = f"{OUT}/runs/ablation_{mob}_noSNR_s{seed}.csv"
        if os.path.exists(f_run): continue
        model, v = train("STGNN", seed, tr, va, log=f"ablation_{mob}_noSNR_s{seed}")
        pd.DataFrame(score(predict(model, te), te, "STGNN w/o SNR features", mob, seed)).to_csv(f_run, index=False)
    del tr, va, te; torch.cuda.empty_cache()
    import glob
    abl = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(f"{OUT}/runs/ablation_*.csv"))])
    abl.to_csv(f"{OUT}/ablation_exp2.csv", index=False)
    display(abl.groupby("method")[["MAE", "RMSE", "R2", "AUC", "AUPRC"]].agg(["mean", "std"]).round(3))

## Figure and LaTeX table

In [ ]:
order = ["LET", "KF-LET", "MLP", "LSTM", "StaticGNN", "STGNN"]
col = {"LET": "#9a9a94", "KF-LET": "#c3c2b7", "MLP": "#eda100", "LSTM": "#eb6834", "StaticGNN": "#1baf7a", "STGNN": "#2a78d6"}
mobs = [m for m in ["rpgm", "gm", "rwp"] if m in res.mobility.unique()]
fig, axs = plt.subplots(1, len(mobs), figsize=(4.2 * len(mobs), 3.8), sharey=True, squeeze=False)
for ax, mob in zip(axs[0], mobs):
    g = res[res.mobility == mob].groupby("method").MAE
    mu, sd = g.mean().reindex(order), g.std().reindex(order).fillna(0)
    ax.bar(range(len(order)), mu, yerr=sd, color=[col[o] for o in order], capsize=3)
    ax.set_xticks(range(len(order)), order, rotation=35, ha="right"); ax.set_title(mob.upper())
    ax.spines[["top", "right"]].set_visible(False); ax.grid(axis="y", alpha=0.25)
axs[0][0].set_ylabel("MAE of residual lifetime [s]")
plt.tight_layout(); fig.savefig(f"{OUT}/fig_exp2_mae.pdf"); fig.savefig(f"{OUT}/fig_exp2_mae.png", dpi=300); plt.show()

names = {"rpgm": "RPGM", "gm": "Gauss--Markov", "rwp": "RWP"}
with open(f"{OUT}/table_exp2.tex", "w") as f:
    for mob in mobs:
        for m in order:
            g = res[(res.mobility == mob) & (res.method == m)]
            if len(g) == 0: continue
            # std over seeds of the per-seed mean (analytical methods: over episodes)
            per = g.groupby("seed")[["MAE", "RMSE", "R2", "AUPRC"]].mean() if g.seed.nunique() > 1 else g[["MAE", "RMSE", "R2", "AUPRC"]]
            c = lambda k, p=2: f"{per[k].mean():.{p}f} $\\pm$ {per[k].std():.{p}f}"
            f.write(f"{names[mob]} & {m} & {c('MAE')} & {c('RMSE')} & {c('R2',3)} & {c('AUPRC',3)} \\\\\n")
print(open(f"{OUT}/table_exp2.tex").read())
import platform, sklearn
json.dump(dict(version="v3", W=W, epochs=EPOCHS, patience=PATIENCE, batch=BATCH, lr=LR, seeds=list(SEEDS), quick=QUICK,
               train_eps=list(TRAIN_EPS), val_eps=list(VAL_EPS), test_eps=list(TEST_EPS), seed0=10_000, kf_grid=KF_GRID,
               gpu=torch.cuda.get_device_name(0) if DEV == "cuda" else "cpu", python=platform.python_version(),
               torch=torch.__version__, numpy=np.__version__, sklearn=sklearn.__version__),
          open(f"{OUT}/config.json", "w"), indent=1)
lat = res.dropna(subset=["latency_ms_median"]).groupby("method")[["params", "batched_ms_per_slot", "latency_ms_median", "latency_ms_p95"]].mean()
display(lat.round(3)); lat.to_csv(f"{OUT}/cost_exp2.csv")
import shutil
shutil.make_archive("exp2_results", "zip", OUT)
try:
    from google.colab import files; files.download("exp2_results.zip")
except ImportError:
    print("exp2_results.zip created")